coverage plot

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def hpd(trace, mass_frac):
    ''' 
    Returns highest probability density region given by a set of samples.

    Parameters
    ----------
    trace: array
        1D array of MCMC samples for a single variable
    mass_frac: float with 0 < mass_frac <= 1
        The fraction of the probability to be included in the HPD. For example, 'massfrac'=0.95 gives a 95% HPD.

    Returns
    -------
    output: array, shape(2,)
        The bounds of the HPD
    '''

    # Get sorted list
    d = np.sort(np.copy(trace))

    # Number of total samples taken
    n = len(trace)

    # Get number of samples that should be included in HPD
    n_samples = np.floor(mass_frac * n).astype(int)

    # Get width (in units of data) of all intervals with n_samples samples
    int_width = d[n_samples:] - d[:n-n_samples]

    # Pick out minimal interval
    min_int = np.argmin(int_width)

    # Return interval
    return np.array([d[min_int], d[min_int+n_samples]])

In [ ]:
# input simulator (to calculate observation), trained posterior model, true_values to be tested

def draw_coverage_plot(simulator, posterior, true_values):
    fractions = np.array([])
    x = np.arange(0, 1, 0.1)
    num_variables = len(true_values[0])


    for i in range(num_variables):
        for j in x:
            contain_true = 0
            for k in true_values:
                samples = posterior.sample((1000,), x = simulator(torch.as_tensor(k)))
                lower_bound, upper_bound = hpd(samples[:,i], x)
                if lower_bound <= j <= upper_bound:
                    contain_true += 1
        
        fractions = np.append(fractions, [contain_true/len(true_values[0])])
    
    plt.plot(x, fractions)
    plt.title("f'{i}")
    plt.show()

In [ ]:
x = np.arange(0, 1, 0.1)
true_values = np.arange(0,10,0.1)
fractions = np.array([])

for i in x:
    lower_bound, upper_bound = hpd(posterior_samples, x)
    contain_true = 0
    for j in true_values:
        if lower_bound <= j <= upper_bound:
            contain_true += 1
    fractions = np.append(fractions, [contain_true/len(true_values)])

plt.plot(x, fractions)
